In [1]:
from datetime import datetime
from dateutil.relativedelta import relativedelta

import pandas as pd
pd.set_option("display.max_columns",30)
import requests

In [2]:
#---
#1.Get data from S3 (taxi, weather)
#2. weather data transformations ----DONE
#3. taxi data transformation ----DONE
#4. update dim_payment_type ----DONE
#5. update dim_company ----DONE
#6. update fact_taxi_trips with the ids from dim_payment_type and dim_company ----DONE
#7. upload dim_weather to S3
#8. upload fact_taxi_trips to S3
#9. upload dim_payment_type and dim_company (current and previous version)
#---


In [3]:
current_datetime=datetime.now() - relativedelta(months=2)
formatted_datetime=current_datetime.strftime("%Y-%m-%d")


url= (
    f"https://data.cityofchicago.org/api/v3/views/ajtu-isnz/query.json?app_token=ogjssL0qdXCJ16jANDq5lFDLc"
    f"&query=select * where trip_start_timestamp>='{formatted_datetime}T00:00:00' AND trip_start_timestamp<='{formatted_datetime}T23:59:59'"

)
response=requests.get(url)
data=response.json()

taxi_trips=pd.DataFrame(data)
taxi_trips.head()

,trip_id,taxi_id,trip_start_timestamp,trip_end_timestamp,trip_seconds,trip_miles,pickup_community_area,dropoff_community_area,fare,tips,tolls,extras,trip_total,payment_type,company,pickup_centroid_latitude,pickup_centroid_longitude,pickup_centroid_location,dropoff_centroid_latitude,dropoff_centroid_longitude,dropoff_centroid_location,:id,:version,:created_at,:updated_at,pickup_census_tract,dropoff_census_tract
0,e05851cbf4fa2744e9158ee4a0b5fefa7aeaaa88,f73d5459dcae455d9d7eb464fb7689b48cf9518d3cf1b7...,2025-10-11T23:45:00.000,2025-10-12T00:00:00.000,367,2.27,76,76,8.5,0,0,20,29,Credit Card,City Service,41.980264315,-87.913624596,"{'type': 'Point', 'coordinates': [-87.91362459...",41.980264315,-87.913624596,"{'type': 'Point', 'coordinates': [-87.91362459...",row-r3t7-nm7h-7kqr,rv-4in4-ni3q-zjqt,2025-11-05T23:18:16.848Z,2025-11-05T23:18:16.848Z,NaN,NaN
1,d6eb0e29eff323b37874d71040bf903b5978394d,8b43f4a681efdaeb470b8570f8f98c822c7ba15a0f740a...,2025-10-11T23:45:00.000,2025-10-12T00:15:00.000,1666,17.56,76,8,43.75,9.85,0,5,59.1,Mobile,Globe Taxi,41.980264315,-87.913624596,"{'type': 'Point', 'coordinates': [-87.91362459...",41.899602111,-87.633308037,"{'type': 'Point', 'coordinates': [-87.63330803...",row-buvs_xqf2-762t,rv-kbtb~ru5u-tb6w,2025-11-05T23:18:16.848Z,2025-11-05T23:18:16.848Z,NaN,NaN
2,d5ca4fdc2436f208fc5ac002afe5cda65e61ea00,c328a43decb00f583c329d91cf0434efdddb3b97f3460c...,2025-10-11T23:45:00.000,2025-10-12T00:00:00.000,867,10.21,76,16,26.5,6.4,0,5,38.4,Credit Card,City Service,41.980264315,-87.913624596,"{'type': 'Point', 'coordinates': [-87.91362459...",41.953582125,-87.72345239,"{'type': 'Point', 'coordinates': [-87.72345239...",row-57i9.23nt.ychm,rv-2eni.9bkc-5wfk,2025-11-05T23:18:16.848Z,2025-11-05T23:18:16.848Z,NaN,NaN
3,d32577dddda768a2d62d27380e1c3d3f63ea886c,73b2f5adecea91eeef3900303a07f1b0519a594cffb6b0...,2025-10-11T23:45:00.000,2025-10-11T23:45:00.000,512,1.45,8,8,8.7,0,0,0,9.2,Mobile,Chicago Taxicab,41.892507781,-87.626214906,"{'type': 'Point', 'coordinates': [-87.62621490...",41.891971508,-87.612945414,"{'type': 'Point', 'coordinates': [-87.61294541...",row-iu3q_rm7m.7nw4,rv-69pb~ihhh~su6j,2025-11-05T23:18:16.848Z,2025-11-05T23:18:16.848Z,17031081500,17031081402
4,d21797be0d35d008afb9298815d466b8efde015b,687e4d96be3cc961be4ce4608d2ad3bfadf54e3bbe80d7...,2025-10-11T23:45:00.000,2025-10-11T23:45:00.000,239,0.8,6,7,5.89,0,0,0,6.39,Mobile,City Service,41.944226601,-87.655998182,"{'type': 'Point', 'coordinates': [-87.65599818...",41.922686284,-87.649488729,"{'type': 'Point', 'coordinates': [-87.64948872...",row-qcd7_3u2z.7pzn,rv-ip6x.u9h9~225i,2025-11-05T23:18:16.848Z,2025-11-05T23:18:16.848Z,NaN,NaN


### Taxi data transformation

In [4]:
def taxi_trips_transformations(taxi_trips: pd.DataFrame) -> pd.DataFrame:
    #---
    #performs transformation on taxi data

    #1. Drop selected columns
    #2. Drop NULL values across columns
    #3. Rename columns
    #4. Create datetime_for_weather helper column (for dim_weather join)
    #
    # :param taxi_trips: dataframe holding daily taxi trips
    #:return:           transformed taxi trips dataframe
    #---

    if not isinstance(taxi_trips, pd.DataFrame):                        #elemi hibakezelés a funkcióban
        raise TypeError("taxi_trips is a not valid pandas Dataframe")
    
    taxi_trips.drop(["pickup_census_tract","dropoff_census_tract","pickup_centroid_location",
                 "dropoff_centroid_location"],axis=1,inplace=True)
    taxi_trips.dropna(inplace=True)   #ha valamelyik rekordban Nan van akkor az egész rekordot törli (adattisztítás)

    taxi_trips.rename(columns={"pickup_community_area":"pickup_community_area_id",
                           "dropoff_community_area":"dropoff_community_area_id"},inplace=True)

    taxi_trips["trip_start_timestamp"]=pd.to_datetime(taxi_trips["trip_start_timestamp"])

    taxi_trips["datetime_for_weather"]=taxi_trips["trip_start_timestamp"].dt.floor("h")   

    return taxi_trips



In [5]:

taxi_trips_transformed=taxi_trips_transformations(taxi_trips)

taxi_trips_transformed.head()

,trip_id,taxi_id,trip_start_timestamp,trip_end_timestamp,trip_seconds,trip_miles,pickup_community_area_id,dropoff_community_area_id,fare,tips,tolls,extras,trip_total,payment_type,company,pickup_centroid_latitude,pickup_centroid_longitude,dropoff_centroid_latitude,dropoff_centroid_longitude,:id,:version,:created_at,:updated_at,datetime_for_weather
0,e05851cbf4fa2744e9158ee4a0b5fefa7aeaaa88,f73d5459dcae455d9d7eb464fb7689b48cf9518d3cf1b7...,2025-10-11 23:45:00,2025-10-12T00:00:00.000,367,2.27,76,76,8.5,0,0,20,29,Credit Card,City Service,41.980264315,-87.913624596,41.980264315,-87.913624596,row-r3t7-nm7h-7kqr,rv-4in4-ni3q-zjqt,2025-11-05T23:18:16.848Z,2025-11-05T23:18:16.848Z,2025-10-11 23:00:00
1,d6eb0e29eff323b37874d71040bf903b5978394d,8b43f4a681efdaeb470b8570f8f98c822c7ba15a0f740a...,2025-10-11 23:45:00,2025-10-12T00:15:00.000,1666,17.56,76,8,43.75,9.85,0,5,59.1,Mobile,Globe Taxi,41.980264315,-87.913624596,41.899602111,-87.633308037,row-buvs_xqf2-762t,rv-kbtb~ru5u-tb6w,2025-11-05T23:18:16.848Z,2025-11-05T23:18:16.848Z,2025-10-11 23:00:00
2,d5ca4fdc2436f208fc5ac002afe5cda65e61ea00,c328a43decb00f583c329d91cf0434efdddb3b97f3460c...,2025-10-11 23:45:00,2025-10-12T00:00:00.000,867,10.21,76,16,26.5,6.4,0,5,38.4,Credit Card,City Service,41.980264315,-87.913624596,41.953582125,-87.72345239,row-57i9.23nt.ychm,rv-2eni.9bkc-5wfk,2025-11-05T23:18:16.848Z,2025-11-05T23:18:16.848Z,2025-10-11 23:00:00
3,d32577dddda768a2d62d27380e1c3d3f63ea886c,73b2f5adecea91eeef3900303a07f1b0519a594cffb6b0...,2025-10-11 23:45:00,2025-10-11T23:45:00.000,512,1.45,8,8,8.7,0,0,0,9.2,Mobile,Chicago Taxicab,41.892507781,-87.626214906,41.891971508,-87.612945414,row-iu3q_rm7m.7nw4,rv-69pb~ihhh~su6j,2025-11-05T23:18:16.848Z,2025-11-05T23:18:16.848Z,2025-10-11 23:00:00
4,d21797be0d35d008afb9298815d466b8efde015b,687e4d96be3cc961be4ce4608d2ad3bfadf54e3bbe80d7...,2025-10-11 23:45:00,2025-10-11T23:45:00.000,239,0.8,6,7,5.89,0,0,0,6.39,Mobile,City Service,41.944226601,-87.655998182,41.922686284,-87.649488729,row-qcd7_3u2z.7pzn,rv-ip6x.u9h9~225i,2025-11-05T23:18:16.848Z,2025-11-05T23:18:16.848Z,2025-10-11 23:00:00


In [6]:
taxi_trips_transformed.info()

<class 'pandas.core.frame.DataFrame'>
Index: 16121 entries, 0 to 17494
Data columns (total 24 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   trip_id                     16121 non-null  object        
 1   taxi_id                     16121 non-null  object        
 2   trip_start_timestamp        16121 non-null  datetime64[ns]
 3   trip_end_timestamp          16121 non-null  object        
 4   trip_seconds                16121 non-null  object        
 5   trip_miles                  16121 non-null  object        
 6   pickup_community_area_id    16121 non-null  object        
 7   dropoff_community_area_id   16121 non-null  object        
 8   fare                        16121 non-null  object        
 9   tips                        16121 non-null  object        
 10  tolls                       16121 non-null  object        
 11  extras                      16121 non-null  object        


### Dim company and dim payment type update

In [7]:
def update_dim_company_dim_payment_type(taxi_trips: pd.DataFrame, dim_df: pd.DataFrame, id_col: str, value_col: str) -> pd.DataFrame:
    #---Extend the dimenion dataframe with new value if has any (generic)
    #:param taxi_trips:         dataframe daily taxi trips
    #:param dim_df:             dataframe dimension data (company, payment)
    #:param id_col:             id columns of dimension dataframe
    #:param value_col           name of column of dimension dataframe contining values
    #:return: extended dimension data
    #---
    todays_dim_data=pd.DataFrame(taxi_trips[value_col].unique(),columns=[value_col])

    new_dim_data=todays_dim_data[~todays_dim_data[value_col].isin(dim_df[value_col])]

    if not new_dim_data.empty:
        max_id=dim_df[id_col].max()
        new_dim_data[id_col]=range(max_id+1,max_id+1+len(new_dim_data))
        dim_df=pd.concat([dim_df,new_dim_data],ignore_index=True)

    return dim_df







In [8]:
dim_payment_type=taxi_trips["payment_type"].drop_duplicates().reset_index(drop=True)
dim_payment_type=pd.DataFrame(
    {"payment_type_id":range(1,len(dim_payment_type)+1),
     "payment_type":dim_payment_type
    }
)

dim_company=taxi_trips["company"].drop_duplicates().reset_index(drop=True)
dim_company=pd.DataFrame(
    {"company_id":range(1,len(dim_company)+1),
     "company":dim_company
    }
)


In [9]:
dim_payment_type_updated=update_dim_company_dim_payment_type(taxi_trips, dim_payment_type, "payment_type_id","payment_type")
dim_company_updated=update_dim_company_dim_payment_type(taxi_trips, dim_company, "company_id","company")

In [10]:
dim_payment_type_updated

,payment_type_id,payment_type
0,1,Credit Card
1,2,Mobile
2,3,Cash
3,4,No Charge
4,5,Prcard
5,6,Unknown
6,7,Dispute


In [11]:
dim_company_updated

,company_id,company
0,1,City Service
1,2,Globe Taxi
2,3,Chicago Taxicab
3,4,5 Star Taxi
4,5,Transit Administrative Center Inc
5,6,Flash Cab
6,7,Taxicab Insurance Agency Llc
7,8,Medallion Leasin
8,9,Taxicab Insurance Agency LLC
9,10,Wolley Taxi


### update fact_taxi_trips with company and payment_type ids

In [12]:
def update_fact_taxi_trips_with_dimension_data(taxi_trips: pd.DataFrame, dim_payment_type: pd.DataFrame,dim_company: pd.DataFrame) ->pd.DataFrame:
    #--Update fact_taxi_trips Dataframe with the dim_company and dim_payment_type ids and delete the string columns (generic)
    #:param taxi_trips:         dataframe daily taxi trips
    #:param dim_payment_type:    payment type master table
    #:param dim_company:         company master table
    #:return: taxi trips data with ids without company and payment type values
    #---
    fact_taxi_trips=taxi_trips.merge(dim_payment_type,on="payment_type")
    fact_taxi_trips=fact_taxi_trips.merge(dim_company,on="company")
    fact_taxi_trips.drop(["payment_type","company"],axis=1,inplace=True)

    return fact_taxi_trips


In [13]:
taxi_trips_transformed_with_dim_ids = update_fact_taxi_trips_with_dimension_data(taxi_trips_transformed, dim_payment_type, dim_company)

taxi_trips_transformed_with_dim_ids.head()

,trip_id,taxi_id,trip_start_timestamp,trip_end_timestamp,trip_seconds,trip_miles,pickup_community_area_id,dropoff_community_area_id,fare,tips,tolls,extras,trip_total,pickup_centroid_latitude,pickup_centroid_longitude,dropoff_centroid_latitude,dropoff_centroid_longitude,:id,:version,:created_at,:updated_at,datetime_for_weather,payment_type_id,company_id
0,e05851cbf4fa2744e9158ee4a0b5fefa7aeaaa88,f73d5459dcae455d9d7eb464fb7689b48cf9518d3cf1b7...,2025-10-11 23:45:00,2025-10-12T00:00:00.000,367,2.27,76,76,8.5,0,0,20,29,41.980264315,-87.913624596,41.980264315,-87.913624596,row-r3t7-nm7h-7kqr,rv-4in4-ni3q-zjqt,2025-11-05T23:18:16.848Z,2025-11-05T23:18:16.848Z,2025-10-11 23:00:00,1,1
1,d6eb0e29eff323b37874d71040bf903b5978394d,8b43f4a681efdaeb470b8570f8f98c822c7ba15a0f740a...,2025-10-11 23:45:00,2025-10-12T00:15:00.000,1666,17.56,76,8,43.75,9.85,0,5,59.1,41.980264315,-87.913624596,41.899602111,-87.633308037,row-buvs_xqf2-762t,rv-kbtb~ru5u-tb6w,2025-11-05T23:18:16.848Z,2025-11-05T23:18:16.848Z,2025-10-11 23:00:00,2,2
2,d5ca4fdc2436f208fc5ac002afe5cda65e61ea00,c328a43decb00f583c329d91cf0434efdddb3b97f3460c...,2025-10-11 23:45:00,2025-10-12T00:00:00.000,867,10.21,76,16,26.5,6.4,0,5,38.4,41.980264315,-87.913624596,41.953582125,-87.72345239,row-57i9.23nt.ychm,rv-2eni.9bkc-5wfk,2025-11-05T23:18:16.848Z,2025-11-05T23:18:16.848Z,2025-10-11 23:00:00,1,1
3,d32577dddda768a2d62d27380e1c3d3f63ea886c,73b2f5adecea91eeef3900303a07f1b0519a594cffb6b0...,2025-10-11 23:45:00,2025-10-11T23:45:00.000,512,1.45,8,8,8.7,0,0,0,9.2,41.892507781,-87.626214906,41.891971508,-87.612945414,row-iu3q_rm7m.7nw4,rv-69pb~ihhh~su6j,2025-11-05T23:18:16.848Z,2025-11-05T23:18:16.848Z,2025-10-11 23:00:00,2,3
4,d21797be0d35d008afb9298815d466b8efde015b,687e4d96be3cc961be4ce4608d2ad3bfadf54e3bbe80d7...,2025-10-11 23:45:00,2025-10-11T23:45:00.000,239,0.8,6,7,5.89,0,0,0,6.39,41.944226601,-87.655998182,41.922686284,-87.649488729,row-qcd7_3u2z.7pzn,rv-ip6x.u9h9~225i,2025-11-05T23:18:16.848Z,2025-11-05T23:18:16.848Z,2025-10-11 23:00:00,2,1
